# WasteSort AI\nClassification automatique des déchets recyclables avec Deep Learning.\n

## 1. Importation des bibliothèques\n

In [ ]:
import os\nimport json\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport tensorflow as tf\nfrom tensorflow.keras.preprocessing.image import ImageDataGenerator\nfrom tensorflow.keras.models import Sequential\nfrom tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout\nfrom tensorflow.keras.callbacks import EarlyStopping\nfrom sklearn.metrics import classification_report, confusion_matrix\n

## 2. Chemin du dataset\nDans Google Colab, place le dataset dans `/content/garbage`.\n

In [ ]:
DATA_DIR = '/content/garbage'\nIMG_SIZE = (150, 150)\nBATCH_SIZE = 32\n

## 3. Compter les images par classe\n

In [ ]:
classes = os.listdir(DATA_DIR)\nfor cls in classes:\n    path = os.path.join(DATA_DIR, cls)\n    if os.path.isdir(path):\n        print(cls, ':', len(os.listdir(path)), 'images')\n

## 4. Prétraitement et data augmentation\n

In [ ]:
datagen = ImageDataGenerator(\n    rescale=1./255,\n    validation_split=0.2,\n    rotation_range=20,\n    zoom_range=0.2,\n    width_shift_range=0.1,\n    height_shift_range=0.1,\n    horizontal_flip=True\n)\n\ntrain_generator = datagen.flow_from_directory(\n    DATA_DIR,\n    target_size=IMG_SIZE,\n    batch_size=BATCH_SIZE,\n    class_mode='categorical',\n    subset='training'\n)\n\nval_generator = datagen.flow_from_directory(\n    DATA_DIR,\n    target_size=IMG_SIZE,\n    batch_size=BATCH_SIZE,\n    class_mode='categorical',\n    subset='validation'\n)\n\nnum_classes = len(train_generator.class_indices)\nprint(train_generator.class_indices)\n

## 5. Visualisation de quelques images\n

In [ ]:
images, labels = next(train_generator)\nplt.figure(figsize=(10, 8))\nfor i in range(9):\n    plt.subplot(3, 3, i+1)\n    plt.imshow(images[i])\n    plt.axis('off')\nplt.show()\n

## 6. Construction du modèle CNN\n

In [ ]:
model = Sequential([\n    Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),\n    MaxPooling2D(2,2),\n    Conv2D(64, (3,3), activation='relu'),\n    MaxPooling2D(2,2),\n    Conv2D(128, (3,3), activation='relu'),\n    MaxPooling2D(2,2),\n    Flatten(),\n    Dense(128, activation='relu'),\n    Dropout(0.5),\n    Dense(num_classes, activation='softmax')\n])\n\nmodel.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])\nmodel.summary()\n

## 7. Entraînement du modèle\n

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)\n\nhistory = model.fit(\n    train_generator,\n    validation_data=val_generator,\n    epochs=15,\n    callbacks=[early_stop]\n)\n

## 8. Courbes Accuracy et Loss\n

In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')\nplt.plot(history.history['val_accuracy'], label='Validation Accuracy')\nplt.legend()\nplt.title('Accuracy')\nplt.show()\n\nplt.plot(history.history['loss'], label='Train Loss')\nplt.plot(history.history['val_loss'], label='Validation Loss')\nplt.legend()\nplt.title('Loss')\nplt.show()\n

## 9. Évaluation du modèle\n

In [ ]:
val_loss, val_accuracy = model.evaluate(val_generator)\nprint('Validation Accuracy:', val_accuracy)\n\npredictions = model.predict(val_generator)\ny_pred = np.argmax(predictions, axis=1)\ny_true = val_generator.classes\nclass_names = list(train_generator.class_indices.keys())\n\nprint(confusion_matrix(y_true, y_pred))\nprint(classification_report(y_true, y_pred, target_names=class_names))\n

## 10. Sauvegarde du modèle\n

In [ ]:
model.save('wastesort_model.keras')\nwith open('class_indices.json', 'w') as f:\n    json.dump(train_generator.class_indices, f)\nprint('Modèle sauvegardé')\n

## 11. Test sur une nouvelle image\n

In [ ]:
from tensorflow.keras.preprocessing import image\n\ndef predict_new_image(image_path):\n    img = image.load_img(image_path, target_size=(150,150))\n    img_array = image.img_to_array(img) / 255.0\n    img_array = np.expand_dims(img_array, axis=0)\n    pred = model.predict(img_array)[0]\n    index = np.argmax(pred)\n    index_to_class = {v:k for k,v in train_generator.class_indices.items()}\n    label = index_to_class[index]\n    confidence = pred[index]\n    plt.imshow(img)\n    plt.axis('off')\n    plt.title(f'{label} - {confidence:.2%}')\n    plt.show()\n\n# Exemple : predict_new_image('/content/test_plastic.jpg')\n